# 1. Dashboard 1

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
DATA_DIR = Path('./data')
# sales.csv là nguồn đã tổng hợp sẵn Revenue & COGS theo ngày
sales = pd.read_csv(DATA_DIR / 'sales.csv', parse_dates=['Date'])
sales['Year']    = sales['Date'].dt.year
sales['Quarter'] = sales['Date'].dt.quarter
sales['Month']   = sales['Date'].dt.month

print(f'Shape      : {sales.shape}')
print(f'Date range : {sales.Date.min().date()} → {sales.Date.max().date()}')

Shape      : (3833, 6)
Date range : 2012-07-04 → 2022-12-31


In [ ]:
total_revenue  = sales['Revenue'].sum()
total_cogs     = sales['COGS'].sum()
gross_profit   = total_revenue - total_cogs
gross_margin   = gross_profit / total_revenue

# CAGR: dùng revenue theo năm
annual = sales.groupby('Year')['Revenue'].sum().reset_index()
annual.columns = ['year', 'revenue']
annual = annual.sort_values('year').reset_index(drop=True)

r_first  = annual['revenue'].iloc[0]
r_last   = annual['revenue'].iloc[-1]
n_years  = annual['year'].iloc[-1] - annual['year'].iloc[0]
cagr     = (r_last / r_first) ** (1 / n_years) - 1

print('=' * 55)
print('Revenue KPIs')
print('=' * 55)
print(f'  Total Revenue    : {total_revenue:>18,.0f} VND')
print(f'  Total COGS       : {total_cogs:>18,.0f} VND')
print(f'  Gross Profit     : {gross_profit:>18,.0f} VND')
print(f'  Gross Margin (%) : {gross_margin*100:>17.2f} %')
print(f'  Revenue CAGR     : {cagr*100:>17.2f} %')
print(f'    ({int(annual["year"].iloc[0])}→{int(annual["year"].iloc[-1])}, {n_years} năm)')
print('=' * 55)

  FINANCIAL KPIs — VinDatathon 2026
  Total Revenue    :     16,430,476,586 VND
  Total COGS       :     14,163,450,519 VND
  Gross Profit     :      2,267,026,066 VND
  Gross Margin (%) :             13.80 %
  Revenue CAGR     :              4.66 %
    (2012→2022, 10 năm)


# 2. Dashboard 2

In [6]:

# ── ORDER & CUSTOMER KPIs ────────────────────────────────────────────────────
orders  = pd.read_csv(DATA_DIR / 'orders.csv', parse_dates=['order_date'], low_memory=False)
reviews = pd.read_csv(DATA_DIR / 'reviews.csv')

total_orders    = orders['order_id'].nunique()
returned_orders = orders[orders['order_status'] == 'returned']['order_id'].nunique()
return_rate     = returned_orders / total_orders

avg_rating    = reviews['rating'].mean()
total_reviews = len(reviews)

print('\n' + '=' * 55)
print('  ORDER & CUSTOMER KPIs')
print('=' * 55)
print(f'  Total Orders       : {total_orders:>12,}')
print(f'  Returned Orders    : {returned_orders:>12,}')
print(f'  Return Rate        : {return_rate*100:>11.5f} %')
print(f'  Total Reviews      : {total_reviews:>12,}')
print(f'  Average Rating     : {avg_rating:>11.2f}')
print('=' * 55)


  ORDER & CUSTOMER KPIs
  Total Orders       :      646,945
  Returned Orders    :       36,142
  Return Rate        :     5.58656 %
  Total Reviews      :      113,551
  Average Rating     :        3.94


# 3.Dashboard 3

In [11]:
orders = pd.read_csv("data/orders.csv", parse_dates=['order_date'], low_memory=False)
payments = pd.read_csv("data/payments.csv")
customers = pd.read_csv("data/customers.csv")

# Preprocessing based on DB3_Customer.ipynb logic
START_DATE = '2012-01-01'
orders = orders[orders['order_date'] >= START_DATE]
valid_order_ids = orders['order_id'].unique()
payments = payments[payments['order_id'].isin(valid_order_ids)]

# Merge for RFM
rfm = orders.merge(payments[['order_id', 'payment_value']], on='order_id', how='left')
rfm = rfm[rfm['order_status'] != 'cancelled']

ref_date = pd.to_datetime('2023-01-01')
rfm_agg = rfm.groupby('customer_id').agg(
last_order_date = ('order_date', 'max'),
first_order_date = ('order_date', 'min'),
frequency = ('order_id', 'nunique'),
monetary = ('payment_value', 'sum'),
avg_order_value = ('payment_value', 'mean')
).reset_index()

rfm_agg['recency_days'] = (ref_date - rfm_agg['last_order_date']).dt.days
rfm_agg['tenure_days'] = (ref_date - rfm_agg['first_order_date']).dt.days

# Handle qcut with duplicates by rank
rfm_agg['R_score'] = pd.qcut(rfm_agg['recency_days'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm_agg['F_score'] = pd.qcut(rfm_agg['frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm_agg['M_score'] = pd.qcut(rfm_agg['monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

def get_rfm_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 4: return 'Champions'
    elif r >= 3 and f >= 3: return 'Loyal Customers'
    elif r >= 4 and f <= 2: return 'New Customers'
    elif r >= 3 and f >= 1: return 'Potential Loyalists'
    elif r <= 2 and f >= 4 and m >= 4: return 'Cant Lose Them'
    elif r <= 2 and f >= 3: return 'At Risk'
    elif r <= 2 and f <= 2: return 'Lost'
    elif r == 3 and f <= 2: return 'About To Sleep'
    else: return 'Need Attention'

rfm_agg['rfm_segment'] = rfm_agg.apply(get_rfm_segment, axis=1)

# Merge back to customers to get full customer list
df = customers.merge(rfm_agg, on='customer_id', how='left')
df['rfm_segment'] = df['rfm_segment'].fillna('Never Purchased')

# Calculate KPIs
# 1. Total Customers
total_customers = df['customer_id'].nunique()
purchased_customers = total_customers - df[df['rfm_segment'] == 'Never Purchased'].shape[0]

# 2. Active Customers
active_segments = ["Champions", "Loyal Customers", "New Customers", "Potential Loyalists", "Cant Lose Them"]
active_customers = df[df['rfm_segment'].isin(active_segments)].shape[0]

# 3. Churn Rate
churn_segments = ["Lost", "At Risk"]
churn_count = df[df['rfm_segment'].isin(churn_segments)].shape[0]
churn_rate_pct = (churn_count / purchased_customers) * 100 if purchased_customers > 0 else 0

# 4. Customer Retention Rate (CRR)
# Using the exact definition from DB3
cs_users = orders[(orders['order_date'] >= '2012-01-01') & (orders['order_date'] < '2022-01-01')]['customer_id'].unique()
CS = len(cs_users)

first_purchase_dates = orders.groupby('customer_id')['order_date'].min()
new_customers_2022 = first_purchase_dates[(first_purchase_dates >= '2022-01-01') & (first_purchase_dates <= '2022-12-31')]
CN = len(new_customers_2022)

ce_users = orders[(orders['order_date'] >= '2022-01-01') & (orders['order_date'] <= '2022-12-31')]['customer_id'].unique()
CE = len(ce_users)

retention_count = CE - CN
crr = (retention_count / CS) * 100 if CS > 0 else 0

# 5. Average LTV
avg_ltv = df[df['rfm_segment'] != "Never Purchased"]['monetary'].mean()

# Print Results
print("\n--- KPI CALCULATOR REPORT ---")
print(f"Total Customers: {total_customers:,}")
print(f"Active Customers: {active_customers:,}")
print(f"Churn Rate: {churn_rate_pct:.2f}%")
print(f"Customer Retention Rate: {crr:.2f}%")
print(f"Average LTV: {avg_ltv:,.2f}")
print("-----------------------------\n")


--- KPI CALCULATOR REPORT ---
Total Customers: 121,930
Active Customers: 55,153
Churn Rate: 37.41%
Customer Retention Rate: 26.29%
Average LTV: 161,522.59
-----------------------------



# 5.Dashboard 5

In [42]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
DATA_DIR = Path('./data')

# 1. LOAD DATA
inventory_df = pd.read_csv(DATA_DIR/'inventory.csv')
products_df = pd.read_csv(DATA_DIR/'products.csv')
order_items_df = pd.read_csv(DATA_DIR/'order_items.csv', low_memory=False)
# Giả định có file orders để lấy ngày tháng lọc sales
orders_df = pd.read_csv(DATA_DIR/'orders.csv', parse_dates=['order_date'])

# Convert date
inventory_df['snapshot_date'] = pd.to_datetime(inventory_df['snapshot_date'])
latest_date = inventory_df['snapshot_date'].max()

# 2. XỬ LÝ INVENTORY
df = inventory_df.merge(products_df[['product_id', 'cogs']], on='product_id', how='left')
df['inventory_value'] = df['stock_on_hand'] * df['cogs']

# Snapshot mới nhất
latest_df = df[df['snapshot_date'] == latest_date].copy()

# 3. XỬ LÝ SALES TRONG 30 NGÀY GẦN NHẤT
start_date = latest_date - pd.Timedelta(days=30)
recent_orders = orders_df[(orders_df['order_date'] > start_date) & (orders_df['order_date'] <= latest_date)]

order_df = order_items_df.merge(recent_orders[['order_id']], on='order_id', how='inner')
order_df = order_df.merge(products_df[['product_id', 'cogs']], on='product_id', how='left')
order_df['cogs_sold'] = order_df['quantity'] * order_df['cogs']

# ----------------- KPIs CALCULATION -----------------

# KPI 1: Inventory Value
inventory_value = latest_df['inventory_value'].sum()

# KPI 2: Stock Available
stock_available = latest_df['stock_on_hand'].sum()

# KPI 3: Average Inventory
avg_inventory = df.groupby('snapshot_date')['inventory_value'].sum().mean()

# KPI 4: Turnover Ratio (Annualized - Vòng quay năm dựa trên 30 ngày gần nhất)
cogs_sold_30d = order_df['cogs_sold'].sum()
turnover = (cogs_sold_30d * 12) / avg_inventory if avg_inventory > 0 else 0

# KPI 5: Inventory to Sales (COGS) Ratio (Tồn kho hiện tại / Doanh số tháng gần nhất)
inventory_to_sales = inventory_value / cogs_sold_30d if cogs_sold_30d > 0 else 0

# KPI 6: Avg Days Supply (Tính toán dựa trên tốc độ bán thực tế)
# Tốc độ bán trung bình mỗi ngày theo từng sản phẩm
daily_sales_qty = order_df.groupby('product_id')['quantity'].sum() / 30
latest_df = latest_df.merge(daily_sales_qty.rename('daily_qty'), on='product_id', how='left').fillna(0)

# Tính số ngày cung ứng = Tồn kho / Tốc độ bán ngày
latest_df['days_of_supply'] = np.where(
    latest_df['daily_qty'] > 0, 
    latest_df['stock_on_hand'] / latest_df['daily_qty'], 
    365
)
latest_df['days_of_supply'] = latest_df['days_of_supply'].clip(upper=365)

avg_days_supply = (
    latest_df['days_of_supply'] * latest_df['stock_on_hand']
).sum() / latest_df['stock_on_hand'].sum() if latest_df['stock_on_hand'].sum() > 0 else 0

# Final KPI Dictionary 
kpi = {
    'Inventory Value': inventory_value,
    'Stock Available': stock_available,
    'Turnover Ratio': turnover,
    'Inventory to Sales (COGS) Ratio': inventory_to_sales,
    'Avg Days Supply': avg_days_supply
}

print("\n--- INVENTORY & LOGISTICS KPIs ---")
for key, value in kpi.items():
    print(f"{key:35}: {value:,.2f}")
print("----------------------------------\n")


--- INVENTORY & LOGISTICS KPIs ---
Inventory Value                    : 387,629,922.58
Stock Available                    : 104,235.00
Turnover Ratio                     : 1.76
Inventory to Sales (COGS) Ratio    : 7.48
Avg Days Supply                    : 297.58
----------------------------------

